# 03 · Preprocessing & Chunking

Clean the extracted Item 1A corpus and split it into model-ready **chunks** (≈ one risk
factor each). The chunking logic lives in `src/preprocess.py`; this notebook runs it and
performs quality-assurance checks.

Output: `data/processed/chunks_clean.csv`.

**Reminder:** feed the `text` column to BERTopic (it needs natural language). `text_clean`
is only for a bag-of-words baseline (LDA/NMF).

In [1]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import preprocess

## 1 · Load inputs and build chunks

In [2]:
corpus, sector_map = preprocess.load_inputs()
print(f"Corpus: {len(corpus)} filings, {corpus['cik'].nunique()} companies, "
      f"{corpus['year'].min()}-{corpus['year'].max()}")

chunks = preprocess.build_chunks(corpus, sector_map)

OUT = preprocess.OUTPUT_CSV
OUT.parent.mkdir(parents=True, exist_ok=True)
chunks.to_csv(OUT, index=False)
print(f"Built {len(chunks)} chunks -> {OUT}")

Corpus: 849 filings, 55 companies, 2010-2025
Built 32409 chunks -> /Users/hdaphne/Desktop/bipm/NLP/topic-modeling-G5/data/processed/chunks_clean.csv


## 2 · Overview

In [3]:
print(f"Chunks          : {len(chunks):,}")
print(f"Filings         : {len(corpus):,}")
print(f"Mean chunks/doc : {len(chunks)/len(corpus):.1f}")
print(f"Companies       : {chunks['cik'].nunique()}")
print(f"Sectors         : {chunks['sector'].nunique()}")
print("\nChunk word-count distribution:")
print(chunks['n_words'].describe().round(1).to_string())

Chunks          : 32,409
Filings         : 849
Mean chunks/doc : 38.2
Companies       : 55
Sectors         : 11

Chunk word-count distribution:
count    32409.0
mean       216.8
std        148.4
min         20.0
25%         67.0
50%        193.0
75%        379.0
max        657.0


## 3 · Quality assurance

These should all pass before topic modelling: no empty text, no out-of-range chunk
lengths, no missing metadata, and no chunk starting mid-sentence.

In [4]:
import re
problems = {}
problems['empty text']            = int((chunks['text'].str.strip() == '').sum())
problems['empty text_clean']      = int((chunks['text_clean'].str.strip() == '').sum())
problems['below MIN_CHUNK_WORDS'] = int((chunks['n_words'] < preprocess.MIN_CHUNK_WORDS).sum())
problems['above MAX_CHUNK_WORDS'] = int((chunks['n_words'] > preprocess.MAX_CHUNK_WORDS).sum())
problems['missing sector']        = int(chunks['sector'].isna().sum())
problems['starts lowercase/punct']= int((~chunks['text'].str.match(r'^[A-Z0-9"]')).sum())

for k, v in problems.items():
    flag = 'OK' if v == 0 else 'CHECK'
    print(f"  [{flag}] {k}: {v}")

  [OK] empty text: 0
  [OK] empty text_clean: 0
  [OK] below MIN_CHUNK_WORDS: 0
  [CHECK] above MAX_CHUNK_WORDS: 50
  [OK] missing sector: 0
  [CHECK] starts lowercase/punct: 3462


## 4 · Chunks per year and per sector

Check temporal density (each year is a time-bin for BERTopic's topics-over-time) and that
no sector is missing. 2026 is expected to be thinner (partial filing cohort).

In [5]:
print('Chunks per year:')
print(chunks.groupby('year').size().to_string())
print('\nChunks per sector:')
print(chunks.groupby('sector').size().sort_values(ascending=False).to_string())

Chunks per year:
year
2010    2436
2011    2790
2012    3164
2013    2706
2014    2058
2015    1971
2016    1833
2017    1816
2018    1863
2019    1811
2020    1525
2021    1639
2022    1647
2023    1626
2024    1735
2025    1789

Chunks per sector:
sector
Financials                4686
Real Estate               4504
Health Care               4345
Utilities                 3552
Information Technology    3533
Consumer Discretionary    2424
Materials                 2268
Communication Services    1930
Industrials               1866
Energy                    1743
Consumer Staples          1558


## 5 · Inspect sample chunks (raw vs. cleaned)

In [6]:
for _, r in chunks.sample(3, random_state=0).iterrows():
    print(f"\n=== {r['ticker']} {r['year']} · chunk {r['chunk_id']} · {r['sector']} ({r['n_words']}w) ===")
    print('TEXT      :', r['text'][:300])
    print('TEXT_CLEAN:', r['text_clean'][:300])


=== SYK 2016 · chunk 1 · Health Care (364w) ===
TEXT      : Other provisions of this legislation, including Medicare provisions aimed at improving quality and decreasing costs, comparative effectiveness research, an independent payment advisory board, and pilot programs to evaluate alternative payment methodologies, could meaningfully change the way healthca
TEXT_CLEAN: provisions legislation including medicare provisions aimed improving quality decreasing costs comparative effectiveness research independent payment advisory board pilot programs evaluate alternative payment methodologies could meaningfully change way healthcare developed delivered cannot predict he

=== AES 2013 · chunk 212 · Utilities (153w) ===
TEXT      : These filings detail the controls IPL plans to add to each of its five baseload units. IPL is seeking and expects to recover through its environmental rate adjustment mechanism all
operating and capital expenditures related to compliance; however, there can be no 

---
Next: keyword-filter these chunks to reputational-risk passages, then run BERTopic
(`src/model.py`).